# Sprint 008: Memory Foundations - End-to-End Test

This notebook tests the Memory Foundations sprint (Sprint 8) end-to-end, specifically testing:
1. **Session Creation**: First execution creates a fresh session automatically
2. **Memory Persistence**: Memory entries from first execution persist across executions
3. **Cross-Execution Reasoning**: Second execution uses memory from first execution to answer related questions

**Test Scenario:**
- **Execution 1**: "Why do dogs chew plastic?" (creates session, stores memory)
- **Execution 2**: "Do cats have the same issue?" (reuses session, accesses memory from execution 1)

**Assumptions:**
- Live llama-cpp backend running on 192.168.128.138:8000
- OpenAI-compatible API endpoint at `http://192.168.128.138:8000/v1/chat/completions`


## Setup and Imports


In [1]:
import json
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional
from IPython.display import display, Markdown, JSON

# Add project root to path
sys.path.insert(0, '/home/brian/projects/Aeon-Architect')

# Import Aeon components
from aeon.llm.adapters.llama_cpp import LlamaCppAdapter
from aeon.memory.stm import STM
from aeon.memory.null_memory import NullMemory
from aeon.session.manager import SessionManager
from aeon.tools.registry import ToolRegistry
from aeon.supervisor.repair import Supervisor
from aeon.validation.schema import Validator
from aeon.kernel.orchestrator import Orchestrator
from aeon.observability.logger import JSONLLogger
from aeon.observability.helpers import generate_correlation_id

print("✓ Imports successful")


✓ Imports successful


## Configuration


In [2]:
# LLM Configuration
LLM_API_URL = "http://192.168.128.138:8000/v1/chat/completions"
LLM_MODEL = "llama-cpp-model"

# Execution Configuration
TTL = 50  # Maximum cycles

# Memory Configuration
STM_CAPACITY = 100  # Maximum entries per session
STM_INITIAL_TTL = 10  # Initial TTL for memory entries
SESSION_INITIAL_TTL = 3  # Initial TTL for sessions (number of executions)

# Logging Configuration
ENABLE_LOGGING = True
LOG_FILE = None  # Set to a file path if you want to save logs

print("✓ Configuration loaded")


✓ Configuration loaded


## Initialize Components


In [3]:
print("Initializing components...")

# Initialize LLM adapter for llama-cpp server
llm = LlamaCppAdapter(
    api_url=LLM_API_URL,
    model=LLM_MODEL,
    max_retries=3
)
print(f"✓ LLM Adapter initialized: {LLM_API_URL}")

# Initialize STM (Short-Term Memory)
stm = STM(capacity=STM_CAPACITY, initial_ttl=STM_INITIAL_TTL)
print(f"✓ STM initialized (capacity={STM_CAPACITY}, initial_ttl={STM_INITIAL_TTL})")

# Initialize Session Manager
session_manager = SessionManager(initial_ttl=SESSION_INITIAL_TTL)
print(f"✓ Session Manager initialized (initial_ttl={SESSION_INITIAL_TTL})")

# Initialize tool registry
tool_registry = ToolRegistry()
print("✓ Tool Registry initialized")

# Initialize supervisor for error repair
supervisor = Supervisor(
    llm_adapter=llm
)
print("✓ Supervisor initialized")

# Initialize validator
validator = Validator()
print("✓ Validator initialized")

# Initialize logger if enabled
logger = None
if ENABLE_LOGGING:
    if LOG_FILE:
        logger = JSONLLogger(file_path=Path(LOG_FILE))
    else:
        logger = JSONLLogger()  # Uses default file
    print("✓ Logger initialized")
else:
    print("⚠ Logging disabled")

print("\n✓ All components initialized successfully!")


Initializing components...
✓ LLM Adapter initialized: http://192.168.128.138:8000/v1/chat/completions
✓ STM initialized (capacity=100, initial_ttl=10)
✓ Session Manager initialized (initial_ttl=3)
✓ Tool Registry initialized
✓ Supervisor initialized
✓ Validator initialized
✓ Logger initialized

✓ All components initialized successfully!


## Create Orchestrator with Memory and Session Support


In [4]:
# Create orchestrator with STM and Session Manager
orchestrator = Orchestrator(
    llm=llm,
    memory_access=stm,  # Use STM for session-scoped memory
    session_manager=session_manager,  # Use Session Manager for session lifecycle
    ttl=TTL,
    tool_registry=tool_registry,
    supervisor=supervisor,
    logger=logger,
)

print("✓ Orchestrator initialized with:")
print(f"  - STM (capacity={STM_CAPACITY}, initial_ttl={STM_INITIAL_TTL})")
print(f"  - Session Manager (initial_ttl={SESSION_INITIAL_TTL})")
print(f"  - TTL={TTL}")


✓ Orchestrator initialized with:
  - STM (capacity=100, initial_ttl=10)
  - Session Manager (initial_ttl=3)
  - TTL=50


## Execution 1: First Question (Creates Fresh Session)

**Question**: "Why do dogs chew plastic?"

This execution will:
1. Create a new session automatically (no session provided)
2. Store memory entries during execution
3. Return the session_id for reuse in execution 2


In [5]:
# First execution - no session provided, orchestrator creates one
request_1 = "Why do dogs chew plastic?"

print(f"🚀 Starting Execution 1...")
print(f"Request: {request_1}")
print(f"Session: Will be created automatically\n")

# Execute first request (no session_id provided, creates new session)
result_1 = orchestrator.execute_multipass(request=request_1)

# Capture session_id from orchestrator
session_id = orchestrator._current_session_id

print(f"\n✓ Execution 1 completed")
print(f"✓ Session ID: {session_id}")

# Verify session was created
if session_id:
    session_state = session_manager.get_session_state(session_id)
    print(f"✓ Session state: {session_state.state}")
    print(f"✓ Session TTL: {session_state.ttl}")
else:
    print("⚠ Warning: No session was created")


🚀 Starting Execution 1...
Request: Why do dogs chew plastic?
Session: Will be created automatically


✓ Execution 1 completed
✓ Session ID: aac50015-b0b1-4431-8e5f-7a0b6aead8b4
✓ Session state: active
✓ Session TTL: 2


## Check Memory After Execution 1

Let's verify that memory entries were stored during execution 1.


In [6]:
if session_id:
    # Get memory entries stored during execution 1
    entries = stm.read_entries(session_id=session_id)
    print(f"📝 Memory entries stored: {len(entries)}")
    
    # Show memory entry metadata (no content for privacy)
    if entries:
        print("\nMemory Entry Metadata:")
        for i, entry in enumerate(entries[:5], 1):  # Show first 5
            print(f"  {i}. Phase: {entry.phase}, Execution: {entry.execution_id}, TTL: {entry.ttl}")
        
        if len(entries) > 5:
            print(f"  ... and {len(entries) - 5} more entries")
    
    # Get usage stats
    stats = stm.get_usage_stats(session_id=session_id)
    print(f"\n📊 Memory Usage Stats:")
    print(f"  - Total entries: {stats.total_entries}")
    print(f"  - Expired entries: {stats.expired_entries}")
    print(f"  - Memory used: {stats.memory_used}")
    print(f"  - Entries considered: {stats.memory_entries_considered}")
    print(f"  - Entries injected: {stats.memory_entries_injected}")
    print(f"  - Failures: {stats.memory_failures}")
else:
    print("⚠ No session available to check memory")


📝 Memory entries stored: 3

Memory Entry Metadata:
  1. Phase: A, Execution: bf65602e-c146-4253-99b7-6530cf56d976, TTL: 6
  2. Phase: B, Execution: bf65602e-c146-4253-99b7-6530cf56d976, TTL: 8
  3. Phase: C, Execution: bf65602e-c146-4253-99b7-6530cf56d976, TTL: 9

📊 Memory Usage Stats:
  - Total entries: 3
  - Expired entries: 0
  - Memory used: True
  - Entries considered: 0
  - Entries injected: 0
  - Failures: 0


## Execution 2: Related Question (Reuses Session)

**Question**: "Do cats have the same issue?"

This execution will:
1. Reuse the session from execution 1
2. Access memory entries from execution 1
3. Use memory context to answer the related question without restating information


## Diagnostic: Test Memory Injection for Phase B

Before Execution 2, let's test what memory would be injected during plan generation (Phase B).

In [7]:
# Test memory injection for Phase B plan generation
if session_id and stm:
    print("=" * 80)
    print("DIAGNOSTIC: Memory Injection Test for Phase B")
    print("=" * 80)
    
    # Simulate Phase B memory injection context
    # This is what would be passed during plan generation
    context = {
        "current_phase": "B",
        "current_execution_id": "test-execution-2",  # Would be actual execution_id
        "injection_point": "PLAN_GENERATION_USER",
    }
    
    # Test select_for_injection directly
    injection_result = stm.select_for_injection(
        session_id=session_id,
        context=context,
        budget=2000,
    )
    
    context_blocks = injection_result.get("context_blocks", [])
    non_authoritative_marker = injection_result.get("non_authoritative_marker", "")
    
    print(f"\n📊 Memory Injection Results:")
    print(f"  - Context blocks found: {len(context_blocks)}")
    print(f"  - Non-authoritative marker: {non_authoritative_marker}")
    
    if context_blocks:
        print(f"\n✅ Memory would be injected! Content:")
        print("-" * 80)
        for i, block in enumerate(context_blocks, 1):
            print(f"\nBlock {i}:")
            print(block)
            print("-" * 80)
        
        # Show what the full memory context would look like
        memory_context = f"{non_authoritative_marker}\n\n"
        memory_context += "\n\n".join(context_blocks)
        memory_context += f"\n\n{non_authoritative_marker}"
        
        print(f"\n📝 Full Memory Context (what would be injected):")
        print("=" * 80)
        print(memory_context)
        print("=" * 80)
    else:
        print(f"\n⚠️  No memory context blocks found!")
        print(f"\nThis means memory injection would NOT happen during plan generation.")
        print(f"\nPossible reasons:")
        print(f"  1. No Phase B entries from Execution 1")
        print(f"  2. Entries expired (TTL <= 0)")
        print(f"  3. Phase filtering excluded entries")
        print(f"  4. Memory entries don't have the expected content structure")
        
        # Show what entries exist
        entries = stm.read_entries(session_id=session_id)
        print(f"\n📋 Available entries in session:")
        for entry in entries:
            print(f"  - Phase: {entry.phase}, Execution: {entry.execution_id[:8]}..., TTL: {entry.ttl}")
else:
    print("⚠ No session_id or STM available for testing")

DIAGNOSTIC: Memory Injection Test for Phase B

📊 Memory Injection Results:
  - Context blocks found: 1
  - Non-authoritative marker: Non-authoritative context from this session (for reference only)

✅ Memory would be injected! Content:
--------------------------------------------------------------------------------

Block 1:
Previous user request: Why do dogs chew plastic?
Previous goal: Understand why dogs chew plastic
Previous steps: Research the common reasons why dogs chew objects, Identify the potential risks associated with dogs chewing plastic, Consult with veterinarians and dog behaviorists to gather expert insights, Analyze the physical and chemical properties of plastic that may make it appealing to dogs, Research dog behavior and learning theories to understand why dogs may chew plastic out of curiosity or instinct
--------------------------------------------------------------------------------

📝 Full Memory Context (what would be injected):
Non-authoritative context from t

In [8]:
# Test prompt registry rendering with memory injection
from aeon.prompts.registry import get_prompt_registry, PromptId, PlanGenerationUserInput
from aeon.tools.registry import ToolRegistry

if session_id and stm:
    print("=" * 80)
    print("DIAGNOSTIC: Prompt Registry Rendering Test")
    print("=" * 80)
    
    # Get prompt registry
    registry = get_prompt_registry()
    
    # Simulate what would be passed during plan generation
    request_2 = "Do cats have the same issue?"
    tool_registry_export = ""  # Empty for this test
    
    input_data = PlanGenerationUserInput(
        request=request_2,
        tool_registry_export=tool_registry_export
    )
    
    # Test WITHOUT memory (baseline)
    print("\n📝 Prompt WITHOUT memory injection:")
    print("-" * 80)
    prompt_no_memory = registry.get_prompt(
        PromptId.PLAN_GENERATION_USER,
        input_data,
        memory=None,  # No memory
        session_id=None,
        execution_id=None,
        phase="B",
    )
    print(prompt_no_memory[:500] + "..." if len(prompt_no_memory) > 500 else prompt_no_memory)
    
    # Test WITH memory (what should happen)
    print("\n\n📝 Prompt WITH memory injection:")
    print("-" * 80)
    
    # Get execution_id from Execution 1 entries
    entries = stm.read_entries(session_id=session_id)
    exec_1_id = entries[0].execution_id if entries else None
    exec_2_id = "test-execution-2"  # Would be actual execution_id
    
    prompt_with_memory = registry.get_prompt(
        PromptId.PLAN_GENERATION_USER,
        input_data,
        memory=stm,  # With memory
        session_id=session_id,
        execution_id=exec_2_id,
        phase="B",
    )
    
    print(prompt_with_memory)
    
    # Check if memory_context was injected
    if "{memory_context}" in prompt_with_memory:
        print("\n⚠️  WARNING: {memory_context} placeholder still present!")
        print("Memory injection may have failed.")
    elif "Previous user request" in prompt_with_memory or "Previous goal" in prompt_with_memory:
        print("\n✅ Memory context was successfully injected!")
        print("\nMemory injection markers found in prompt.")
    else:
        print("\n⚠️  No memory context markers found.")
        print("Memory may not have been injected, or no relevant entries found.")
else:
    print("⚠ No session_id or STM available for testing")

DIAGNOSTIC: Prompt Registry Rendering Test

📝 Prompt WITHOUT memory injection:
--------------------------------------------------------------------------------
Generate a plan to accomplish the following request:

Do cats have the same issue?



Return a JSON plan with goal and steps.


📝 Prompt WITH memory injection:
--------------------------------------------------------------------------------
Generate a plan to accomplish the following request:

Do cats have the same issue?

Non-authoritative context from this session (for reference only)

Previous user request: Why do dogs chew plastic?
Previous goal: Understand why dogs chew plastic
Previous steps: Research the common reasons why dogs chew objects, Identify the potential risks associated with dogs chewing plastic, Consult with veterinarians and dog behaviorists to gather expert insights, Analyze the physical and chemical properties of plastic that may make it appealing to dogs, Research dog behavior and learning theories to unde

In [9]:
# Second execution - reuse the session from execution 1
request_2 = "Do cats have the same issue?"

print(f"🚀 Starting Execution 2...")
print(f"Request: {request_2}")
print(f"Session: {session_id} (reusing from Execution 1)\n")

# Execute second request with session_id parameter to reuse the session
# The orchestrator now supports an optional session_id parameter
result_2 = orchestrator.execute_multipass(request=request_2, session_id=session_id)

print(f"\n✓ Execution 2 completed")
print(f"✓ Session ID: {orchestrator._current_session_id}")

# Verify session was reused
if orchestrator._current_session_id == session_id:
    print("✅ Session successfully reused!")
    session_state = session_manager.get_session_state(session_id)
    print(f"✓ Session state: {session_state.state}")
    print(f"✓ Session TTL: {session_state.ttl} (should be decremented)")
else:
    print(f"⚠ Warning: Session ID changed ({session_id} -> {orchestrator._current_session_id})")


🚀 Starting Execution 2...
Request: Do cats have the same issue?
Session: aac50015-b0b1-4431-8e5f-7a0b6aead8b4 (reusing from Execution 1)


✓ Execution 2 completed
✓ Session ID: aac50015-b0b1-4431-8e5f-7a0b6aead8b4
✅ Session successfully reused!
✓ Session state: active
✓ Session TTL: 1 (should be decremented)


## Check Memory After Execution 2

Let's verify that memory from execution 1 was accessed and new entries were stored.


## Diagnostic: Memory Content Inspection

Let's inspect what content is actually stored in memory entries and what would be extracted for injection.


In [10]:
# Diagnostic: Show what's stored in Phase B entries from Execution 1
if session_id:
    entries = stm.read_entries(session_id=session_id)
    
    # Find Phase B entries from Execution 1 (first execution)
    if entries:
        first_exec_id = entries[0].execution_id
        exec_1_entries = [e for e in entries if e.execution_id == first_exec_id]
        phase_b_entries = [e for e in exec_1_entries if e.phase == "B"]
    else:
        phase_b_entries = []
    
    if phase_b_entries:
        print("=" * 80)
        print("DIAGNOSTIC: Phase B Entry Content from Execution 1")
        print("=" * 80)
        
        for i, entry in enumerate(phase_b_entries, 1):
            print(f"\nEntry {i}:")
            print(f"  Phase: {entry.phase}")
            print(f"  Execution ID: {entry.execution_id}")
            print(f"  TTL: {entry.ttl}")
            print(f"  Content keys: {list(entry.content.keys()) if isinstance(entry.content, dict) else 'N/A'}")
            
            if isinstance(entry.content, dict):
                print(f"\n  Stored content:")
                for key, value in entry.content.items():
                    if key == "step_descriptions" and isinstance(value, list):
                        print(f"    {key}: {len(value)} step(s)")
                        for j, step in enumerate(value[:3], 1):  # Show first 3
                            print(f"      {j}. {step}")
                        if len(value) > 3:
                            print(f"      ... and {len(value) - 3} more")
                    else:
                        # Truncate long values
                        value_str = str(value)
                        if len(value_str) > 100:
                            value_str = value_str[:100] + "..."
                        print(f"    {key}: {value_str}")
                
                # Simulate what would be extracted
                print(f"\n  What would be extracted for injection:")
                block_parts = []
                
                # Extract user request
                user_request = entry.content.get("user_request") or entry.content.get("user_prompt", "")
                if user_request:
                    block_parts.append(f"Previous user request: {user_request}")
                
                # Extract goal
                goal = entry.content.get("goal", "")
                if goal:
                    block_parts.append(f"Previous goal: {goal}")
                
                # Extract step descriptions
                step_descriptions = entry.content.get("step_descriptions", [])
                if step_descriptions:
                    if isinstance(step_descriptions, list):
                        descs = [str(s) for s in step_descriptions if s]
                        if descs:
                            block_parts.append(f"Previous steps: {', '.join(descs)}")
                    else:
                        block_parts.append(f"Previous steps: {step_descriptions}")
                
                if block_parts:
                    formatted = "\n".join(block_parts)
                    print(f"    {formatted}")
                else:
                    print("    ⚠ No extractable fields found - would fall back to JSON dump")
    else:
        print("⚠ No Phase B entries found from Execution 1")
else:
    print("⚠ No session available")


DIAGNOSTIC: Phase B Entry Content from Execution 1

Entry 1:
  Phase: B
  Execution ID: df9592b7-2087-405f-b9ef-f9c5b66b294f
  TTL: 8
  Content keys: ['user_request', 'goal', 'step_descriptions']

  Stored content:
    user_request: Do cats have the same issue?
    goal: Determine if cats have the same issue
    step_descriptions: 4 step(s)
      1. Look up general information about common health issues in cats
      2. Analyze research findings to identify potential health issues that affect cats
      3. Research and compare common health issues in other animals (e.g. dogs, birds) to see if any similarities exist
      ... and 1 more

  What would be extracted for injection:
    Previous user request: Do cats have the same issue?
Previous goal: Determine if cats have the same issue
Previous steps: Look up general information about common health issues in cats, Analyze research findings to identify potential health issues that affect cats, Research and compare common health issues in 

In [11]:
if session_id:
    # Get all memory entries (from both executions)
    entries = stm.read_entries(session_id=session_id)
    print(f"📝 Total memory entries: {len(entries)}")
    
    # Group by execution
    from collections import defaultdict
    by_execution = defaultdict(list)
    for entry in entries:
        by_execution[entry.execution_id].append(entry)
    
    print(f"\n📊 Entries by execution:")
    for exec_id, exec_entries in by_execution.items():
        print(f"  - {exec_id}: {len(exec_entries)} entries")
        phases = [e.phase for e in exec_entries]
        phase_counts = {p: phases.count(p) for p in set(phases)}
        print(f"    Phases: {phase_counts}")
    
    # Get usage stats
    stats = stm.get_usage_stats(session_id=session_id)
    print(f"\n📊 Memory Usage Stats (after Execution 2):")
    print(f"  - Total entries: {stats.total_entries}")
    print(f"  - Expired entries: {stats.expired_entries}")
    print(f"  - Memory used: {stats.memory_used}")
    print(f"  - Entries considered: {stats.memory_entries_considered}")
    print(f"  - Entries injected: {stats.memory_entries_injected}")
    print(f"  - Failures: {stats.memory_failures}")
    
    # Verify memory was used (entries should have been considered/injected)
    if stats.memory_entries_considered > 0 or stats.memory_entries_injected > 0:
        print("\n✅ Memory was successfully accessed during Execution 2!")
    else:
        print("\n⚠ Warning: Memory may not have been accessed")
else:
    print("⚠ No session available to check memory")


📝 Total memory entries: 3

📊 Entries by execution:
  - df9592b7-2087-405f-b9ef-f9c5b66b294f: 3 entries
    Phases: {'C': 1, 'A': 1, 'B': 1}

📊 Memory Usage Stats (after Execution 2):
  - Total entries: 3
  - Expired entries: 0
  - Memory used: True
  - Entries considered: 3
  - Entries injected: 3
  - Failures: 0

✅ Memory was successfully accessed during Execution 2!


## Display Results

Let's see the final answers from both executions to verify that Execution 2 used context from Execution 1.


In [12]:
# Display Execution 1 Result
print("=" * 80)
print("EXECUTION 1 RESULT")
print("=" * 80)
print(f"\nQuestion: {request_1}\n")

# Extract and display final answer clearly
if result_1 and 'final_answer' in result_1:
    final_answer = result_1['final_answer']
    
    # Handle both dict and object formats
    if isinstance(final_answer, dict):
        answer_text = final_answer.get('answer_text', 'No answer text available')
        confidence = final_answer.get('confidence', None)
    elif hasattr(final_answer, 'answer_text'):
        answer_text = final_answer.answer_text
        confidence = getattr(final_answer, 'confidence', None)
    else:
        answer_text = str(final_answer)
        confidence = None
    
    # Display answer prominently
    print("Final Answer:")
    print("-" * 80)
    display(Markdown(f"**{answer_text}**"))
    print("-" * 80)
    
    # Show confidence if available
    if confidence is not None:
        print(f"\nConfidence: {confidence}")
    
    # Show memory usage stats if available
    if isinstance(final_answer, dict) and 'metadata' in final_answer:
        metadata = final_answer['metadata']
        if 'memory_used' in metadata or 'memory_entries_injected' in metadata:
            print("\nMemory Usage:")
            if 'memory_used' in metadata:
                print(f"  - Memory used: {metadata['memory_used']}")
            if 'memory_entries_injected' in metadata:
                print(f"  - Entries injected: {metadata['memory_entries_injected']}")
    
    # Now display the full result details
    print("\n" + "=" * 80)
    print("FULL EXECUTION DETAILS")
    print("=" * 80)
    display(JSON(result_1))
else:
    print("⚠ No final_answer found in result")
    if result_1:
        print("\nAvailable keys in result:")
        for key in result_1.keys():
            print(f"  - {key}")
        print("\nFull result:")
        display(JSON(result_1))
    else:
        print("⚠ No result available")


EXECUTION 1 RESULT

Question: Why do dogs chew plastic?

Final Answer:
--------------------------------------------------------------------------------


**Dogs chew plastic due to various reasons, including curiosity, instinct, and boredom. While plastic is not a natural material for dogs to chew on, some dogs may find it appealing due to its texture, smell, or taste. However, chewing on plastic can be harmful to dogs, as it can cause digestive issues, choking hazards, and other health problems. Therefore, it is essential to provide dogs with suitable chew toys and to supervise their behavior to prevent accidents.**

--------------------------------------------------------------------------------

Confidence: 0.8

FULL EXECUTION DETAILS


<IPython.core.display.JSON object>

In [13]:
# Display Execution 2 Result
print("=" * 80)
print("EXECUTION 2 RESULT")
print("=" * 80)
print(f"\nQuestion: {request_2}")
print("(This should reference information from Execution 1 without restating it)\n")

# Extract and display final answer clearly
if result_2 and 'final_answer' in result_2:
    final_answer = result_2['final_answer']
    
    # Handle both dict and object formats
    if isinstance(final_answer, dict):
        answer_text = final_answer.get('answer_text', 'No answer text available')
        confidence = final_answer.get('confidence', None)
    elif hasattr(final_answer, 'answer_text'):
        answer_text = final_answer.answer_text
        confidence = getattr(final_answer, 'confidence', None)
    else:
        answer_text = str(final_answer)
        confidence = None
    
    # Display answer prominently
    print("Final Answer:")
    print("-" * 80)
    display(Markdown(f"**{answer_text}**"))
    print("-" * 80)
    
    # Show confidence if available
    if confidence is not None:
        print(f"\nConfidence: {confidence}")
    
    # Check if answer references dogs/plastic (from execution 1)
    answer_text_lower = answer_text.lower()
    if 'dog' in answer_text_lower or 'plastic' in answer_text_lower:
        print("\n✅ Answer references context from Execution 1!")
    else:
        print("\n⚠ Note: Answer may not explicitly reference Execution 1 context")
    
    # Show memory usage stats if available
    if isinstance(final_answer, dict) and 'metadata' in final_answer:
        metadata = final_answer['metadata']
        if 'memory_used' in metadata or 'memory_entries_injected' in metadata:
            print("\nMemory Usage:")
            if 'memory_used' in metadata:
                print(f"  - Memory used: {metadata['memory_used']}")
            if 'memory_entries_considered' in metadata:
                print(f"  - Entries considered: {metadata['memory_entries_considered']}")
            if 'memory_entries_injected' in metadata:
                print(f"  - Entries injected: {metadata['memory_entries_injected']}")
            if 'memory_failures' in metadata:
                print(f"  - Failures: {metadata['memory_failures']}")
    
    # Now display the full result details
    print("\n" + "=" * 80)
    print("FULL EXECUTION DETAILS")
    print("=" * 80)
    display(JSON(result_2))
else:
    print("⚠ No final_answer found in result")
    if result_2:
        print("\nAvailable keys in result:")
        for key in result_2.keys():
            print(f"  - {key}")
        print("\nFull result:")
        display(JSON(result_2))
    else:
        print("⚠ No result available")


EXECUTION 2 RESULT

Question: Do cats have the same issue?
(This should reference information from Execution 1 without restating it)

Final Answer:
--------------------------------------------------------------------------------


**The task requires a moderate level of reasoning depth to understand the context and the issue at hand. Information sufficiency is high because the task is relatively specific and can be answered with a yes or no. Expected tool usage is moderate because the task does not require specialized knowledge or tools, but may require some basic research. Output breadth is moderate because the task requires a response that is concise but informative. Confidence requirement is medium because the task requires some level of confidence in the answer, but not to the extent that it requires expert-level knowledge or absolute certainty.**

--------------------------------------------------------------------------------

Confidence: 0.5

⚠ Note: Answer may not explicitly reference Execution 1 context

FULL EXECUTION DETAILS


<IPython.core.display.JSON object>

## Verify Memory Injection

Let's check if memory was actually injected into prompts during Execution 2 by examining the execution history.


In [14]:
# Check execution history for memory usage indicators
if result_2 and 'execution_history' in result_2:
    history = result_2['execution_history']
    
    print("🔍 Checking Execution 2 for memory injection indicators...\n")
    
    # Look for memory usage in Phase E metadata
    if 'final_answer' in result_2:
        final_answer = result_2['final_answer']
        if hasattr(final_answer, 'metadata') and final_answer.metadata:
            metadata = final_answer.metadata
            print("Phase E Metadata (Memory Usage):")
            if 'memory_used' in metadata:
                print(f"  - Memory used: {metadata['memory_used']}")
            if 'memory_entries_considered' in metadata:
                print(f"  - Entries considered: {metadata['memory_entries_considered']}")
            if 'memory_entries_injected' in metadata:
                print(f"  - Entries injected: {metadata['memory_entries_injected']}")
            if 'memory_failures' in metadata:
                print(f"  - Failures: {metadata['memory_failures']}")
            
            # Verify memory was used
            if metadata.get('memory_used', False) or metadata.get('memory_entries_injected', 0) > 0:
                print("\n✅ Memory was successfully injected into prompts!")
            else:
                print("\n⚠ Warning: Memory may not have been injected")
    
    # Check execution passes for memory indicators
    if 'execution_passes' in history:
        passes = history['execution_passes']
        print(f"\n📊 Execution Passes: {len(passes)}")
        for i, pass_data in enumerate(passes[:3], 1):  # Show first 3 passes
            if 'phase' in pass_data:
                print(f"  Pass {i}: Phase {pass_data['phase']}")
else:
    print("⚠ No execution history available")


🔍 Checking Execution 2 for memory injection indicators...



## Test Summary

This test verifies:
1. ✅ **Session Creation**: Execution 1 automatically created a session
2. ✅ **Memory Storage**: Memory entries were stored during Execution 1
3. ✅ **Session Reuse**: Execution 2 reused the session from Execution 1
4. ✅ **Memory Access**: Memory from Execution 1 was accessed during Execution 2
5. ✅ **Cross-Execution Reasoning**: Execution 2 should reference context from Execution 1

**Expected Behavior:**
- Execution 2's answer should reference dogs/plastic (from Execution 1) when answering about cats
- The answer should demonstrate that the LLM understood the connection between the two questions
- Memory usage stats should show entries were considered and injected


In [15]:
# Final summary
print("=" * 80)
print("TEST SUMMARY")
print("=" * 80)

print(f"\n✅ Session Management:")
print(f"  - Session ID: {session_id}")
if session_id:
    session_state = session_manager.get_session_state(session_id)
    print(f"  - Session State: {session_state.state}")
    print(f"  - Session TTL: {session_state.ttl}")

print(f"\n✅ Memory Statistics:")
if session_id:
    stats = stm.get_usage_stats(session_id=session_id)
    print(f"  - Total Entries: {stats.total_entries}")
    print(f"  - Memory Used: {stats.memory_used}")
    print(f"  - Entries Considered: {stats.memory_entries_considered}")
    print(f"  - Entries Injected: {stats.memory_entries_injected}")

print(f"\n✅ Test Status:")
if session_id and stats.memory_entries_injected > 0:
    print("  ✅ PASS: Memory was successfully used across executions")
else:
    print("  ⚠ WARNING: Memory may not have been fully utilized")

print("\n" + "=" * 80)


TEST SUMMARY

✅ Session Management:
  - Session ID: aac50015-b0b1-4431-8e5f-7a0b6aead8b4
  - Session State: active
  - Session TTL: 1

✅ Memory Statistics:
  - Total Entries: 3
  - Memory Used: True
  - Entries Considered: 3
  - Entries Injected: 3

✅ Test Status:
  ✅ PASS: Memory was successfully used across executions

